# Rozdział 5 — Filtracja cyfrowa i zmiana częstotliwości próbkowania

Notebook wykonawczy do `manuscript/chapters/ch05_filtracja_resampling.md`.
E5.1–E5.13 generują osobne figury PNG/SVG i wyniki liczbowe. Tekst rozdziału
zawiera wszystkie wyjaśnienia konieczne do interpretacji. Źródła: kanoniczny
materiał M4 oraz `M4_filtracja_i_resampling_v.3.ipynb`.

Dane są całkowicie syntetyczne; zakresy EKG są przykładami dydaktycznymi,
nie specyfikacją urządzenia klinicznego. Nie są potrzebne dane zewnętrzne ani sieć.
Należy zachować katalog `src/` z pakietu. Uruchomienie: katalog główny pakietu lub
`notebooks/`, Python 3.12 z zależnościami `requirements_ch05.txt`.
Można wykonać wszystkie komórki także przez `python scripts/run_ch05.py`.
Nie osadzamy obrazów w base64; figurom odpowiadają zwykłe ścieżki względne.


In [1]:
from pathlib import Path
import sys,json,platform,warnings
import numpy as np
import scipy
from scipy import signal
import matplotlib
import matplotlib.pyplot as plt
ROOT=next((p for p in [Path.cwd(),Path.cwd().parent] if (p/'src/bspsbook/filtering.py').is_file()),None)
if ROOT is None:raise RuntimeError('Uruchom notebook z katalogu pakietu lub notebooks/.')
sys.path.insert(0,str(ROOT/'src'))
from bspsbook.plotting import configure_plots,save_figure
from bspsbook.signals import synthetic_ecg
from bspsbook.filtering import (db_response,group_delay_sos,filter_blocks,fft_mask,
                                tone_fit,psd,band_power,preservation_metrics)
configure_plots(chapter=5)
FIG=ROOT/'figures/generated'
CHECKS=[];RESULTS={};SEED=2026
def check(label,condition):
    if not bool(condition):raise AssertionError(label)
    CHECKS.append(label);print('OK:',label)
ENV={'python':platform.python_version(),'numpy':np.__version__,'scipy':scipy.__version__,
     'matplotlib':matplotlib.__version__,'seed':SEED,'rng':'numpy.random.default_rng / PCG64'}
FS=1000.
t,clean,noisy,events,components=synthetic_ecg(fs=FS,seed=SEED)
print(json.dumps(ENV,ensure_ascii=False,indent=2))
print('EKG: N=',len(t),'fs=',FS,'mV; zdarzenia:',len(events))


{
  "python": "3.12.14",
  "numpy": "2.3.5",
  "scipy": "1.17.0",
  "matplotlib": "3.10.8",
  "seed": 2026,
  "rng": "numpy.random.default_rng / PCG64"
}
EKG: N= 12000 fs= 1000.0 mV; zdarzenia: 14


## E5.1 — gładszy nie oznacza lepiej zachowany; rys. 5.1

Wspólne EKG: fs=1000 Hz, 12 s, 72 min⁻¹, morfologia pięciu Gaussianów;
dryft 0.18 mV przy 0.25 Hz i 0.05 mV przy 0.06 Hz; sieć 50 Hz, 0.08 mV;
szum normalny SD=0.028 mV, seed=2026. Parametry morfologii są w generatorze
i tabeli w rozdziale. Wariant A: Butterworth parametr N=4, BP 0.7–12 Hz,
przód–tył; B: N=4, BP 0.7–35 Hz, przyczynowo, stan zerowy.
W obu filtrach BP rzeczywisty rząd wynosi 8. Cechy oceniane przy znanych zdarzeniach,
po pominięciu 1 s na każdym brzegu. Nie uruchamiamy pełnego detektora R.

![Rys. 5.1](../figures/generated/fig_05_01_ecg_tradeoff.png)


In [2]:
sosa=signal.butter(4,[.7,12],btype='bandpass',fs=FS,output='sos')
sosb=signal.butter(4,[.7,35],btype='bandpass',fs=FS,output='sos')
ya=signal.sosfiltfilt(sosa,noisy);yb=signal.sosfilt(sosb,noisy)
rows={name:preservation_metrics(clean,y,events,FS) for name,y in [('Zakłócony',noisy),('A',ya),('B',yb)]}
fig,ax=plt.subplots(2,1,figsize=(9,6),layout='constrained')
for name,y,ls in [('Odniesienie',clean,'-'),('Zakłócony',noisy,':')]:
    ax[0].plot(t,y,ls,lw=1,label=name)
ax[0].set(xlim=(.4,3.2),xlabel='Czas [s]',ylabel='Amplituda [mV]',title='a) Czysta morfologia i kontrolowane zakłócenia')
ax[0].legend()
at=events[4]
for name,y,ls in [('Odniesienie',clean,'-'),('A: wąskie pasmo, przód–tył',ya,'--'),('B: szersze pasmo, przyczynowo',yb,'-.')]:
    ax[1].plot((t-at)*1000,y,ls,lw=1.5,label=name)
ax[1].set(xlim=(-120,150),xlabel='Czas względem zdarzenia [ms]',ylabel='Amplituda [mV]',title='b) Tłumienie amplitudy i przesunięcie są różnymi błędami')
ax[1].legend()
check('E5.1: wąskie pasmo bardziej zmienia amplitudę R',rows['A']['R_amplitude_error_pct']>rows['B']['R_amplitude_error_pct'])
check('E5.1: filtr przyczynowy opóźnia maksimum R',rows['B']['R_time_signed_mean_ms']>5)
RESULTS['E5.1']=rows
print(json.dumps(rows,ensure_ascii=False,indent=2))
save_figure(fig,FIG,'fig_05_01_ecg_tradeoff')


OK: E5.1: wąskie pasmo bardziej zmienia amplitudę R
OK: E5.1: filtr przyczynowy opóźnia maksimum R
{
  "Zakłócony": {
    "R_amplitude_error_pct": 10.055541235883924,
    "R_time_MAE_ms": 1.5,
    "R_time_signed_mean_ms": 0.0,
    "R_time_signed_SD_ms": 1.9578900207451218,
    "QRS_NRMSE_aligned": 0.11819123539226056,
    "beats": 12,
    "RMSE_center_mV": 0.14665592067353783,
    "RMSE_start_mV": 0.20249439925383125,
    "RMSE_end_mV": 0.1513336868864812
  },
  "A": {
    "R_amplitude_error_pct": 55.75219689710652,
    "R_time_MAE_ms": 0.9166666666666666,
    "R_time_signed_mean_ms": -0.9166666666666666,
    "R_time_signed_SD_ms": 0.2763853991962833,
    "QRS_NRMSE_aligned": 0.16029657201789113,
    "beats": 12,
    "RMSE_center_mV": 0.12779005287660045,
    "RMSE_start_mV": 0.09504295382428714,
    "RMSE_end_mV": 0.14027336287804337
  },
  "B": {
    "R_amplitude_error_pct": 9.26570924694047,
    "R_time_MAE_ms": 11.75,
    "R_time_signed_mean_ms": 11.75,
    "R_time_signed_SD_ms": 0

## E5.2 — splot, LTI i średnia ruchoma; rys. 5.2

x=[1,2,0,-1], h=[0.25,0.5,0.25]; pełny splot i stan zerowy.
Numeryczna kontrola superpozycji i przesunięcia o 7 próbek przy danych z otoczeniem zerowym.
Średnie ruchome M=5,21,81 stosowane przyczynowo do tego samego zaszumionego EKG.
Suma współczynników=1 nie zapewnia zachowania krótkiego R.

![Rys. 5.2](../figures/generated/fig_05_02_convolution.png)


In [3]:
x=np.array([1.,2.,0.,-1.]);h=np.array([.25,.5,.25])
manual=np.zeros(len(x)+len(h)-1)
for i in range(len(x)):
    for j in range(len(h)):manual[i+j]+=x[i]*h[j]
check('E5.2: pełny splot ręczny = numpy',np.allclose(manual,np.convolve(x,h)))
check('E5.2: wynik referencyjny splotu',np.allclose(manual,[.25,1.,1.25,.25,-.5,-.25]))
imp=np.zeros(50);imp[0]=1;imp7=np.zeros(50);imp7[7]=1
h0=signal.lfilter(h,[1],imp);h7=signal.lfilter(h,[1],imp7)
check('E5.2: przesunięcie wejścia przesuwa odpowiedź',np.allclose(h7[7:],h0[:-7]))
check('E5.2: superpozycja',np.allclose(signal.lfilter(h,[1],2*imp-3*imp7),2*h0-3*h7))
check('E5.2: lfilter FIR = początek pełnego splotu',np.allclose(signal.lfilter(h,[1],x),manual[:len(x)]))
fig,ax=plt.subplot_mosaic([['h','y'],['ecg','ecg']],figsize=(9.1,6.1),layout='constrained')
ax['h'].stem(np.arange(len(h)),h,basefmt=' ')
ax['h'].set(xlabel='Indeks k',ylabel='Waga h[k] [1]',title='a) Odpowiedź impulsowa')
ax['y'].stem(np.arange(len(manual)),manual,basefmt=' ')
ax['y'].set(xlabel='Indeks n',ylabel='y[n] [j.u.]',title='b) Pełny splot: N+L−1 próbek')
at=events[4]
ax['ecg'].plot((t-at)*1000,clean,color='black',lw=1.5,label='Czyste odniesienie')
rows=[]
for M,ls in [(5,'-'),(21,'--'),(81,'-.')]:
    out=signal.lfilter(np.ones(M)/M,[1],noisy)
    ax['ecg'].plot((t-at)*1000,out,ls,lw=1.1,label=f'M={M}, opóźnienie {(M-1)/2:g} ms')
    rows.append({'M':M,'delay_samples':(M-1)/2,'metrics':preservation_metrics(clean,out,events,FS)})
ax['ecg'].set(xlim=(-100,170),xlabel='Czas względem zdarzenia [ms]',ylabel='Amplituda [mV]',title='c) Jednostkowe przenoszenie stałej nie chroni morfologii QRS')
ax['ecg'].legend()
RESULTS['E5.2']={'full_convolution':manual.tolist(),'moving_averages':rows}
save_figure(fig,FIG,'fig_05_02_convolution')


OK: E5.2: pełny splot ręczny = numpy
OK: E5.2: wynik referencyjny splotu
OK: E5.2: przesunięcie wejścia przesuwa odpowiedź
OK: E5.2: superpozycja
OK: E5.2: lfilter FIR = początek pełnego splotu
Figura: figures/generated/fig_05_02_convolution.png (+ SVG)


## E5.3 — specyfikacja i odpowiedź częstotliwościowa; rys. 5.3

D2: sinusoidy 1,10,50,80 Hz, amplitudy po 1 j.u., fs=1000 Hz, T=8 s.
Demonstracyjne filtry Butterwortha: LP20, HP30, BP5–60, BS45–55 Hz,
parametr N=4. W BP/BS rzeczywisty rząd=8, w LP/HP=4.
Miary eksperymentalne: jeden przebieg przyczynowy, pominięto po 2 s;
porównanie amplitud tonów z |H|, bez mylenia jednego przebiegu z |H|².

![Rys. 5.3](../figures/generated/fig_05_03_filter_types.png)


In [4]:
fd=np.array([1.,10.,50.,80.]);td=np.arange(8000)/FS
xd=np.sin(2*np.pi*fd[:,None]*td).sum(axis=0)
fig,ax=plt.subplots(figsize=(9.0,4.2),layout='constrained')
rows=[]
specs=[('LP',20.,'lowpass','-'),('HP',30.,'highpass','--'),('BP',[5.,60.],'bandpass','-.'),('BS',[45.,55.],'bandstop',':')]
for name,cut,kind,ls in specs:
    sos=signal.butter(4,cut,btype=kind,fs=FS,output='sos')
    f,H=signal.sosfreqz(sos,worN=32768,fs=FS)
    cutlabel='–'.join(f'{v:g}' for v in cut) if isinstance(cut,list) else f'{cut:g}'
    ax.plot(f,db_response(H),ls,lw=1.5,label=f'{name}: {cutlabel} Hz')
    _,H_at=signal.sosfreqz(sos,worN=fd,fs=FS)
    out=signal.sosfilt(sos,xd)
    amps=[tone_fit(out[2000:-2000],FS,fr,2.) for fr in fd]
    # LP/HP/BS szybko wygaszają stan przejściowy; BP o dolnej granicy 5 Hz też mieści się w marginesie.
    check(f'E5.3: amplitudy tonów zgodne z jednym |H|, {name}',np.allclose(amps,np.abs(H_at),atol=.005,rtol=.02))
    rows.append({'type':name,'prototype_order':4,'realized_order':8 if kind in ['bandpass','bandstop'] else 4,
                 'gain_dB_at_1_10_50_80':db_response(H_at).tolist(),'measured_amplitudes':amps})
for fr in fd:ax.axvline(fr,color='black',ls=':',lw=.7)
ax.set(xlim=(0,110),ylim=(-85,3),xlabel='Częstotliwość [Hz]',ylabel='Wzmocnienie [dB]',title='Cztery różne decyzje o tych samych składowych')
ax.legend(ncol=2)
RESULTS['E5.3']=rows
save_figure(fig,FIG,'fig_05_03_filter_types')


OK: E5.3: amplitudy tonów zgodne z jednym |H|, LP
OK: E5.3: amplitudy tonów zgodne z jednym |H|, HP
OK: E5.3: amplitudy tonów zgodne z jednym |H|, BP
OK: E5.3: amplitudy tonów zgodne z jednym |H|, BS
Figura: figures/generated/fig_05_03_filter_types.png (+ SVG)


## E5.4 — FIR metodą okien; rys. 5.4

LP35 Hz przy fs=1000 Hz. Porównanie L=31,101,401 z symetrycznym oknem Hamminga;
przy L=101 porównanie boxcar, hann, hamming, blackman. Okna projektowe są
symetryczne, w odróżnieniu od okresowych okien PSD z rozdziału 4.
Ręczna konstrukcja sinc × okno, normalizacja sumy do 1, kontrola zgodności z firwin.
Kontrola stałego opóźnienia dla FIR101 w paśmie 1–20 Hz.

![Rys. 5.4](../figures/generated/fig_05_04_fir.png)


In [5]:
fig,ax=plt.subplots(1,2,figsize=(9.7,4.1),layout='constrained')
rows=[]
for L,ls in [(31,'-'),(101,'--'),(401,'-.')]:
    b=signal.firwin(L,35,fs=FS,window='hamming')
    f,H=signal.freqz(b,worN=65536,fs=FS);db=db_response(H)
    ax[0].plot(f,db,ls,label=f'L={L}; d={(L-1)/2:g} ms')
    rows.append({'L':L,'order':L-1,'delay_ms':(L-1)/2,
                 'max_abs_gain_0_20_dB':float(np.max(abs(db[f<=20]))),
                 'max_gain_above80_dB':float(np.max(db[f>=80]))})
for win,ls in [('boxcar','-'),('hann','--'),('hamming','-.'),('blackman',':')]:
    b=signal.firwin(101,35,fs=FS,window=win)
    n=np.arange(101);hd=2*35/FS*np.sinc(2*35/FS*(n-50))
    hand=hd*signal.get_window(win,101,fftbins=False);hand/=hand.sum()
    check(f'E5.4: sinc × symetryczne okno = firwin, {win}',np.allclose(hand,b,rtol=1e-13,atol=1e-14))
    f,H=signal.freqz(b,worN=65536,fs=FS)
    ax[1].plot(f,db_response(H),ls,label=win)
b101=signal.firwin(101,35,fs=FS,window='hamming')
probe=np.linspace(1,20,200)
_,gd=signal.group_delay((b101,[1]),w=probe,fs=FS)
check('E5.4: symetria FIR101',np.allclose(b101,b101[::-1]))
check('E5.4: opóźnienie FIR101 = 50 próbek',np.allclose(gd,50,atol=1e-10))
for a in ax:a.set(xlim=(0,110),ylim=(-110,3),xlabel='Częstotliwość [Hz]',ylabel='Wzmocnienie [dB]');a.legend()
ax[0].set_title('a) Węższe przejście kosztem długości')
ax[1].set_title('b) Okno rozdziela błąd między pasmami')
RESULTS['E5.4']={'length_comparison':rows,'delay_samples_FIR101':float(np.median(gd))}
save_figure(fig,FIG,'fig_05_04_fir')


OK: E5.4: sinc × symetryczne okno = firwin, boxcar
OK: E5.4: sinc × symetryczne okno = firwin, hann
OK: E5.4: sinc × symetryczne okno = firwin, hamming
OK: E5.4: sinc × symetryczne okno = firwin, blackman
OK: E5.4: symetria FIR101
OK: E5.4: opóźnienie FIR101 = 50 próbek
Figura: figures/generated/fig_05_04_fir.png (+ SVG)


## E5.5 — IIR: rodzina a wspólne wymagania; rys. 5.5

Najpierw LP Butterworth i Czebyszew I, oba rzędu 4, Wn=35 Hz, rp=0.5 dB.
Następnie wspólna specyfikacja: fp=30 Hz, fst=50 Hz, Ap=1 dB, As=40 dB,
fs=1000 Hz. Rząd z buttord/cheb1ord, projekt bezpośrednio output='sos'.
Kontrola pełnych pasm na gęstej siatce obejmującej dokładne granice 30 i 50 Hz.
Opóźnienie grupowe z pochodnej fazy SOS, w próbkach i przeliczone na ms.

![Rys. 5.5](../figures/generated/fig_05_05_iir.png)


In [6]:
fig,ax=plt.subplots(3,1,figsize=(9,8.0),layout='constrained')
same=[];common=[]
for name,sos,ls in [('Butterworth',signal.butter(4,35,fs=FS,output='sos'),'-'),
                    ('Czebyszew I',signal.cheby1(4,.5,35,fs=FS,output='sos'),'--')]:
    f,H=signal.sosfreqz(sos,worN=65536,fs=FS)
    ax[0].plot(f,db_response(H),ls,label=name)
    fg=np.linspace(1,30,2000);gd=group_delay_sos(sos,fg,FS)
    ax[2].plot(fg,1000*gd/FS,ls,label=name+' (rząd 4)')
    same.append({'name':name,'order':4,'sections':len(sos),'group_delay_1_30_ms_range':[float(1000*gd.min()/FS),float(1000*gd.max()/FS)]})
for name,func in [('Butterworth',signal.buttord),('Czebyszew I',signal.cheb1ord)]:
    order,wn=func(30,50,1,40,fs=FS)
    sos=(signal.butter(order,wn,fs=FS,output='sos') if name=='Butterworth'
         else signal.cheby1(order,1,wn,fs=FS,output='sos'))
    f=np.unique(np.r_[np.linspace(0,499.999,65536),30.,50.])
    _,H=signal.sosfreqz(sos,worN=f,fs=FS);db=db_response(H)
    loss=float(-db[f<=30].min());stop=float(-db[f>=50].max())
    check(f'E5.5: pasmo przepustowe <=1 dB, {name}',loss<=1+1e-8)
    check(f'E5.5: pasmo zaporowe >=40 dB, {name}',stop>=40-1e-8)
    poles=signal.sos2zpk(sos)[1]
    check(f'E5.5: bieguny wewnątrz okręgu jednostkowego, {name}',np.max(abs(poles))<1)
    ax[1].plot(f,db,'-' if name=='Butterworth' else '--',label=f'{name}: rząd {order}, SOS={len(sos)}')
    common.append({'name':name,'order':int(order),'Wn_Hz':float(wn),'sections':len(sos),
                   'max_pass_loss_dB':loss,'min_stop_atten_dB':stop,'max_pole_radius':float(np.max(abs(poles)))})
for a in ax[:2]:a.set(xlim=(0,100),ylim=(-85,3),xlabel='Częstotliwość [Hz]',ylabel='Wzmocnienie [dB]');a.legend()
for val in [-1,-40]:ax[1].axhline(val,color='black',ls=':',lw=.8)
for val in [30,50]:ax[1].axvline(val,color='black',ls=':',lw=.8)
ax[0].set_title('a) Ten sam rząd i ta sama liczba Wn — inne konwencje granicy')
ax[1].set_title('b) Wspólna specyfikacja — inny wymagany rząd')
ax[2].set(xlabel='Częstotliwość [Hz]',ylabel='Opóźnienie grupowe [ms]',title='c) Zmienna faza pozostaje częścią kompromisu')
ax[2].legend()
RESULTS['E5.5']={'same_order':same,'common_specification':common}
print(json.dumps(common,ensure_ascii=False,indent=2))
save_figure(fig,FIG,'fig_05_05_iir')


OK: E5.5: pasmo przepustowe <=1 dB, Butterworth
OK: E5.5: pasmo zaporowe >=40 dB, Butterworth
OK: E5.5: bieguny wewnątrz okręgu jednostkowego, Butterworth
OK: E5.5: pasmo przepustowe <=1 dB, Czebyszew I
OK: E5.5: pasmo zaporowe >=40 dB, Czebyszew I
OK: E5.5: bieguny wewnątrz okręgu jednostkowego, Czebyszew I
[
  {
    "name": "Butterworth",
    "order": 11,
    "Wn_Hz": 31.888012229762356,
    "sections": 6,
    "max_pass_loss_dB": 1.0000000000000506,
    "min_stop_atten_dB": 43.44556373566317,
    "max_pole_radius": 0.9720664204770827
  },
  {
    "name": "Czebyszew I",
    "order": 6,
    "Wn_Hz": 30.0,
    "sections": 3,
    "max_pass_loss_dB": 0.9999999999999445,
    "min_stop_atten_dB": 45.71092740202979,
    "max_pole_radius": 0.9884145879311079
  }
]
Figura: figures/generated/fig_05_05_iir.png (+ SVG)


## E5.6 — stabilny model a realizacja numeryczna; rys. 5.6

Butterworth LP35 Hz, rząd 12, fs=1000 Hz. Projekt w float64;
jednorazowe zaokrąglenie współczynników do float32, lfilter b,a vs sosfilt SOS.
Wejście: 1500 próbek szumu SD=0.1 j.u., seed=2026. To celowy przypadek
niepowodzenia, nie twierdzenie o każdym filtrze w postaci b,a.
Wykres pokazuje logarytm modułu i przycina wartości >10⁶ wyłącznie do wizualizacji.

![Rys. 5.6](../figures/generated/fig_05_06_numeric_stability.png)


In [7]:
rng=np.random.default_rng(SEED)
xs=rng.normal(0,.1,1500).astype(np.float32)
b,a=signal.butter(12,35,fs=FS);sos=signal.butter(12,35,fs=FS,output='sos')
with warnings.catch_warnings(), np.errstate(over='ignore',invalid='ignore'):
    warnings.simplefilter('ignore',RuntimeWarning)
    direct=signal.lfilter(b.astype(np.float32),a.astype(np.float32),xs)
    stable=signal.sosfilt(sos.astype(np.float32),xs)
stable64=signal.sosfilt(sos,xs.astype(float))
check('E5.6: wynik SOS float32 skończony',np.isfinite(stable).all())
check('E5.6: SOS float32 bliski odniesieniu float64',np.sqrt(np.mean((stable-stable64)**2))<1e-5)
finite=np.isfinite(direct)
peak=float(np.max(np.abs(direct[finite]))) if finite.any() else None
check('E5.6: forma b,a ujawnia duży błąd w tym przypadku',not finite.all() or peak>100)
fig,ax=plt.subplots(1,2,figsize=(9.4,4.1),layout='constrained')
shown=np.clip(np.nan_to_num(np.abs(direct.astype(float)),nan=1e6,posinf=1e6,neginf=1e6),1e-10,1e6)
ax[0].semilogy(np.arange(len(xs))/FS,shown,'--',label='b,a float32 (sufit wykresu 10⁶)')
ax[0].semilogy(np.arange(len(xs))/FS,np.maximum(abs(stable),1e-10),label='SOS float32')
ax[0].set(xlabel='Czas [s]',ylabel='|y| [j.u.]',title='a) Różne skutki zaokrąglenia współczynników')
ax[0].legend(fontsize=7)
for pole,ls in [(.8,'-'),(.98,'--'),(1.02,'-.')]:
    n=np.arange(160);ax[1].semilogy(n,pole**n,ls,label=f'a={pole:g}')
ax[1].set(xlabel='Indeks n',ylabel='|h[n]| [1]',title='b) Pojedynczy biegun: zanik i wzrost')
ax[1].legend()
RESULTS['E5.6']={'direct_all_finite':bool(finite.all()),'direct_finite_count':int(finite.sum()),
    'direct_finite_peak':peak,'SOS_all_finite':bool(np.isfinite(stable).all()),
    'SOS32_vs64_RMSE':float(np.sqrt(np.mean((stable-stable64)**2)))}
save_figure(fig,FIG,'fig_05_06_numeric_stability')


OK: E5.6: wynik SOS float32 skończony
OK: E5.6: SOS float32 bliski odniesieniu float64
OK: E5.6: forma b,a ujawnia duży błąd w tym przypadku
Figura: figures/generated/fig_05_06_numeric_stability.png (+ SVG)


## E5.7 — jeden przebieg a przód–tył; rys. 5.7

Ten sam Butterworth BP0.7–35 Hz, N=4 prototypu (rzeczywisty rząd 8).
sosfilt przy stanie zerowym oraz sosfiltfilt padtype='odd', padlen=27.
Charakterystyka teoretyczna |H| i |H|² odnosi się do wnętrza długiego zapisu;
nie traktujemy całej skończonej operacji brzegowej jako jednego układu LTI.
Kontrola sinusoidy 35 Hz na osobnym zapisie 60 s z marginesem 15 s.

![Rys. 5.7](../figures/generated/fig_05_07_forward_backward.png)


In [8]:
sos=signal.butter(4,[.7,35],btype='bandpass',fs=FS,output='sos')
causal=signal.sosfilt(sos,noisy)
fb=signal.sosfiltfilt(sos,noisy,padtype='odd',padlen=27)
f=np.geomspace(.1,120,6000);_,H=signal.sosfreqz(sos,worN=f,fs=FS)
fig,ax=plt.subplots(2,1,figsize=(9,6.2),layout='constrained')
ax[0].semilogx(f,db_response(H),label='Jeden przebieg')
ax[0].semilogx(f,2*db_response(H),'--',label='Przód–tył: |H|²')
for freq in [.7,35]:ax[0].axvline(freq,color='black',ls=':',lw=.8)
ax[0].set(ylim=(-85,3),xlabel='Częstotliwość [Hz]',ylabel='Wzmocnienie [dB]',title='a) Usunięcie fazy zmienia także amplitudę')
ax[0].legend()
at=events[4]
for label,y,ls in [('Odniesienie',clean,'-'),('Przyczynowo',causal,'--'),('Przód–tył',fb,'-.')]:
    ax[1].plot((t-at)*1000,y,ls,label=label)
ax[1].set(xlim=(-110,140),xlabel='Czas względem zdarzenia [ms]',ylabel='Amplituda [mV]',title='b) Zerowa faza nie zapewnia zachowania amplitudy')
ax[1].legend()
tt=np.arange(60000)/FS; tone=np.sin(2*np.pi*35*tt)
one=signal.sosfilt(sos,tone);two=signal.sosfiltfilt(sos,tone,padtype='odd',padlen=27)
_,hh=signal.sosfreqz(sos,worN=[35.],fs=FS)
A1=tone_fit(one[15000:-15000],FS,35,15);A2=tone_fit(two[15000:-15000],FS,35,15)
check('E5.7: amplituda tonu jednego przebiegu = |H|',np.isclose(A1,abs(hh[0]),atol=1e-6))
check('E5.7: amplituda tonu przód–tył = |H|²',np.isclose(A2,abs(hh[0])**2,atol=1e-6))
RESULTS['E5.7']={'at35Hz_single_dB':float(db_response(hh)[0]),'at35Hz_fb_dB':float(2*db_response(hh)[0]),
    'measured_tone_amplitudes':[A1,A2],'causal':preservation_metrics(clean,causal,events,FS),
    'forward_backward':preservation_metrics(clean,fb,events,FS),'padtype':'odd','padlen':27}
save_figure(fig,FIG,'fig_05_07_forward_backward')


OK: E5.7: amplituda tonu jednego przebiegu = |H|
OK: E5.7: amplituda tonu przód–tył = |H|²
Figura: figures/generated/fig_05_07_forward_backward.png (+ SVG)


## E5.8 — stan i brakujące otoczenie zapisu; rys. 5.8

LP Butterworth35 Hz, rząd 4. Stała: stan zerowy vs stan ustalony sosfilt_zi.
EKG w blokach 250 próbek: stan przenoszony vs zerowany.
Wycinek [0.55,3.20) s: filtrowanie po wycięciu (odd, padlen=15; lub padlen=0)
vs wycięcie po filtrowaniu pełnych 12 s. Błąd brzegu oceniany w pierwszych/
ostatnich 0.25 s, środek pozostały. Nie uznajemy żadnego paddingu za prawdziwe dane.

![Rys. 5.8](../figures/generated/fig_05_08_state_edges.png)


In [9]:
sos=signal.butter(4,35,fs=FS,output='sos')
constant=np.ones(1000);z=signal.sosfilt(sos,constant)
matched,_=signal.sosfilt(sos,constant,zi=signal.sosfilt_zi(sos))
check('E5.8: stan ustalony LP zachowuje stałą',np.allclose(matched,1,atol=1e-12))
whole=signal.sosfilt(sos,noisy)
kept=filter_blocks(sos,noisy,250,True);reset=filter_blocks(sos,noisy,250,False)
check('E5.8: stan przenoszony = cały zapis',np.allclose(kept,whole,rtol=0,atol=1e-14))
check('E5.8: zerowanie stanu daje sztuczne granice',np.max(abs(reset-whole))>.1)
i0,i1=550,3200;xc=noisy[i0:i1];tc=t[i0:i1]
reference=signal.sosfiltfilt(sos,noisy,padtype='odd',padlen=15)[i0:i1]
padded=signal.sosfiltfilt(sos,xc,padtype='odd',padlen=15)
unpad=signal.sosfiltfilt(sos,xc,padlen=0)
rows=[]
for label,y in [('odd15',padded),('no_pad',unpad)]:
    err=y-reference
    rows.append({'variant':label,'RMSE_start_mV':float(np.sqrt(np.mean(err[:250]**2))),
                 'RMSE_middle_mV':float(np.sqrt(np.mean(err[250:-250]**2))),
                 'RMSE_end_mV':float(np.sqrt(np.mean(err[-250:]**2)))})
check('E5.8: końcowy brzeg różni się od środka',rows[1]['RMSE_end_mV']>rows[1]['RMSE_middle_mV']+1e-4)
fig,ax=plt.subplots(3,1,figsize=(9,7.4),layout='constrained')
ax[0].plot(np.arange(150)/FS,z[:150],label='Stan zerowy');ax[0].plot(np.arange(150)/FS,matched[:150],'--',label='Stan ustalony')
ax[0].set(xlabel='Czas [s]',ylabel='Wartość [j.u.]',title='a) Rozruch przy stałym wejściu')
ax[0].legend()
sl=slice(1670,1930)
ax[1].plot(t[sl],whole[sl],label='Cały zapis');ax[1].plot(t[sl],kept[sl],'--',label='Stan przenoszony');ax[1].plot(t[sl],reset[sl],':',label='Stan zerowany')
ax[1].axvline(1.75,color='black',ls=':',lw=.8)
ax[1].set(xlabel='Czas [s]',ylabel='Amplituda [mV]',title='b) Granica bloków jest częścią realizacji')
ax[1].legend()
ax[2].plot(tc,padded-reference,label='Wycięcie → filtracja, odd15')
ax[2].plot(tc,unpad-reference,'--',label='Wycięcie → filtracja, bez paddingu')
ax[2].set(xlabel='Czas [s]',ylabel='Różnica [mV]',title='c) Odniesienie: filtracja pełnego zapisu → wycięcie')
ax[2].legend()
RESULTS['E5.8']={'block_kept_max_error':float(np.max(abs(kept-whole))),
    'block_reset_RMSE_mV':float(np.sqrt(np.mean((reset-whole)**2))),'crop':rows}
save_figure(fig,FIG,'fig_05_08_state_edges')


OK: E5.8: stan ustalony LP zachowuje stałą
OK: E5.8: stan przenoszony = cały zapis
OK: E5.8: zerowanie stanu daje sztuczne granice
OK: E5.8: końcowy brzeg różni się od środka
Figura: figures/generated/fig_05_08_state_edges.png (+ SVG)


## E5.9 — notch: szerokość i niedostrojenie; rys. 5.9

Notch f0=50 Hz, Q=5,15,30,60; iirnotch → tf2sos (tylko sekcja rzędu 2).
Charakterystyka jednego przebiegu, działanie na danych przez sosfiltfilt,
odd i padlen=9. Kontrolny ton 49.6 Hz: 30 s, brzegi po 5 s pominięte.
Koszt na czystym EKG jest mierzony oddzielnie; nie nazywamy całego
sygnału w paśmie 49–51 Hz zakłóceniem. Porównanie z BS45–55 Hz (rzeczywisty rząd 8)
zapisano w wynikach liczbowych, bez nadawania mu statusu lepszej metody.

![Rys. 5.9](../figures/generated/fig_05_09_notch.png)


In [10]:
fig,ax=plt.subplots(1,2,figsize=(9.7,4.2),layout='constrained')
rows=[]
tt=np.arange(30000)/FS;interference=np.sin(2*np.pi*49.6*tt)
for Q,ls in [(5,'-'),(15,'--'),(30,'-.'),(60,':')]:
    b,a=signal.iirnotch(50,Q,fs=FS);sos=signal.tf2sos(b,a)
    f=np.linspace(44,56,10001);_,H=signal.sosfreqz(sos,worN=f,fs=FS)
    ax[0].plot(f,db_response(H),ls,label=f'Q={Q}, BW={50/Q:.2f} Hz')
    out=signal.sosfiltfilt(sos,interference,padtype='odd',padlen=9)
    amp=tone_fit(out[5000:-5000],FS,49.6,5)
    _,hf=signal.sosfreqz(sos,worN=[49.6],fs=FS)
    theory=float(40*np.log10(abs(hf[0])))
    measured=float(20*np.log10(amp))
    check(f'E5.9: niedostrojenie zgodne z |H|², Q={Q}',abs(measured-theory)<.03)
    cost=preservation_metrics(clean,signal.sosfiltfilt(sos,clean,padtype='odd',padlen=9),events,FS)
    rows.append({'Q':Q,'nominal_BW_Hz':50/Q,'atten49_6_measured_dB':measured,'atten49_6_theory_dB':theory,'clean_cost':cost})
ax[0].set(xlabel='Częstotliwość [Hz]',ylabel='Wzmocnienie jednego przebiegu [dB]',ylim=(-65,2),title='a) Wysokie Q zwęża ingerencję')
ax[0].legend(fontsize=7)
ax[1].plot([r['Q'] for r in rows],[r['atten49_6_measured_dB'] for r in rows],'o-')
ax[1].set(xlabel='Dobroć Q',ylabel='Zmiana amplitudy tonu [dB]',title='b) Ten sam notch 50 Hz wobec tonu 49,6 Hz')
check('E5.9: wysoki Q słabiej tłumi niedostrojony ton',rows[-1]['atten49_6_measured_dB']>rows[0]['atten49_6_measured_dB'])
soswide=signal.butter(4,[45,55],btype='bandstop',fs=FS,output='sos')
widecost=preservation_metrics(clean,signal.sosfiltfilt(soswide,clean,padtype='odd',padlen=27),events,FS)
RESULTS['E5.9']={'notches':rows,'wide_bandstop_clean_cost':widecost}
save_figure(fig,FIG,'fig_05_09_notch')


OK: E5.9: niedostrojenie zgodne z |H|², Q=5
OK: E5.9: niedostrojenie zgodne z |H|², Q=15
OK: E5.9: niedostrojenie zgodne z |H|², Q=30
OK: E5.9: niedostrojenie zgodne z |H|², Q=60
OK: E5.9: wysoki Q słabiej tłumi niedostrojony ton
Figura: figures/generated/fig_05_09_notch.png (+ SVG)


## E5.10 — maska FFT, dzwonienie i splot cykliczny; rys. 5.10

Skok: fs=1000 Hz, N=4096, początek n=1500; maska 35 Hz, maska z przejściem
30–40 Hz i liniowy splot z FIR401 Hamming35. Impuls przy końcu:
N=2048, jedynki n=1958…1977, maska 35 Hz, FFT długości N i 8N.
Kontrola symetrii maski oraz liniowego splotu przez FFT z długością >=N+L−1.
Dopełnienie dla stałego FIR nie jest tym samym co przebudowa maski przy innej FFT.

![Rys. 5.10](../figures/generated/fig_05_10_fft_mask.png)


In [11]:
step=np.zeros(4096);step[1500:]=1
sharp,imag,_,_=fft_mask(step,FS,35)
smooth,_,_,_=fft_mask(step,FS,30,40)
bf=signal.firwin(401,35,fs=FS,window='hamming')
fir=signal.fftconvolve(step,bf,mode='same')
check('E5.10: symetryczna maska daje wynik rzeczywisty',imag<1e-12)
overs={name:float(100*(y[1100:1900].max()-1)) for name,y in [('sharp',sharp),('smooth',smooth),('FIR401',fir)]}
xx=np.zeros(2048);xx[1958:1978]=1
cyc,_,_,_=fft_mask(xx,FS,35)
pad,_,_,_=fft_mask(xx,FS,35,nfft=8*len(xx))
wrap0=float(np.sqrt(np.mean(cyc[:250]**2)));wrap8=float(np.sqrt(np.mean(pad[:250]**2)))
check('E5.10: padding ogranicza zawinięcie w przykładzie',wrap8<wrap0/3)
NFFT=len(xx)+len(bf)-1
yfft=np.fft.ifft(np.fft.fft(xx,NFFT)*np.fft.fft(bf,NFFT)).real
linear=np.convolve(xx,bf,mode='full')
check('E5.10: FFT z właściwą długością = splot liniowy FIR',np.allclose(yfft,linear,atol=1e-12))
fig,ax=plt.subplots(2,1,figsize=(9,6.3),layout='constrained')
tt=np.arange(len(step))/FS
for label,y,ls in [('Skok',step,'-'),('Maska prostokątna',sharp,'--'),('Przejście 30–40 Hz',smooth,'-.'),('FIR401 Hamming',fir,':')]:
    ax[0].plot((tt-1.5)*1000,y,ls,lw=1.3,label=label)
ax[0].set(xlim=(-150,200),xlabel='Czas względem skoku [ms]',ylabel='Wartość [j.u.]',title='a) Łagodne pasmo nie gwarantuje małego pierwszego nadstrzału')
ax[0].legend(fontsize=7)
tx=np.arange(len(xx))/FS
ax[1].plot(tx,cyc,label='DFT długości N');ax[1].plot(tx,pad,'--',label='Maska po dopełnieniu 8N')
ax[1].set(xlim=(0,.25),ylim=(-.035,.035),xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='b) Początek zapisu: powiększona skala amplitudy')
ax[1].legend()
RESULTS['E5.10']={'overshoot_pct':overs,'RMS_first250ms_N':wrap0,'RMS_first250ms_8N':wrap8,'linear_FFT_max_error':float(np.max(abs(yfft-linear)))}
save_figure(fig,FIG,'fig_05_10_fft_mask')


OK: E5.10: symetryczna maska daje wynik rzeczywisty
OK: E5.10: padding ogranicza zawinięcie w przykładzie
OK: E5.10: FFT z właściwą długością = splot liniowy FIR
Figura: figures/generated/fig_05_10_fft_mask.png (+ SVG)


## E5.11 — antyaliasing przed redukcją; rys. 5.11

fs_in=1000 Hz, fs_out=200 Hz, M=5, T=5 s. Sinusoidy: 20 Hz A=1,
180 Hz A=0.6, obie faza 0. Wybór co piątej próbki daje dokładnie 0.4*sin(20 Hz).
Jawny wariant offline: liniowy splot 'same' z symetrycznym FIR401 Hamming90 Hz,
potem redukcja; porównanie resample_poly(1,5), window=('kaiser',5), padtype='constant'.
Amplitudy oceniane po 0.5 s marginesu. Osobno EKG1000→250 Hz z tonem180 Hz.

![Rys. 5.11](../figures/generated/fig_05_11_decimation.png)


In [12]:
ti=np.arange(5000)/FS
useful=np.sin(2*np.pi*20*ti);hi=.6*np.sin(2*np.pi*180*ti);mix=useful+hi
naive=mix[::5];reference=useful[::5]
ba=signal.firwin(401,90,fs=FS,window='hamming')
explicit=signal.fftconvolve(mix,ba,mode='same')[::5]
poly=signal.resample_poly(mix,1,5,window=('kaiser',5.),padtype='constant')
check('E5.11: alias 180 Hz = −20 Hz i amplituda 0,4',np.allclose(naive,.4*reference,atol=1e-11))
amps={name:tone_fit(y[100:-100],200,20,.5) for name,y in [('Referencja',reference),('Co 5.',naive),('FIR → co 5.',explicit),('resample_poly',poly)]}
check('E5.11: jawny antyaliasing zachowuje ton20 Hz',abs(amps['FIR → co 5.']-1)<.01)
check('E5.11: resample_poly zachowuje ton20 Hz',abs(amps['resample_poly']-1)<.01)
_,h100=signal.freqz(ba,worN=[100.,180.],fs=FS)
fig,ax=plt.subplots(1,2,figsize=(9.6,4.0),layout='constrained')
to=np.arange(len(reference))/200
for name,y,ls in [('Odniesienie',reference,'-'),('Co 5.',naive,'--'),('FIR → co 5.',explicit,'-.'),('resample_poly',poly,':')]:
    ax[0].plot(to,y,ls,label=name)
ax[0].set(xlim=(1,1.2),xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='a) Alias znosi część informacji użytecznej')
ax[0].legend(fontsize=7)
ax[1].bar(range(len(amps)),list(amps.values()),color=['0.2','#B25324','#176A8A','#25805D'])
ax[1].set_xticks(range(len(amps)),list(amps),rotation=15,ha='right')
ax[1].set(ylabel='Amplituda 20 Hz [j.u.]',title='b) Ten sam ton po różnych procedurach',ylim=(0,1.15))
# Stres-test biomedyczny, nie dodajemy tonu 180 Hz do wcześniejszych eksperymentów.
stress=noisy+.12*np.sin(2*np.pi*180*t)
naive_ecg=stress[::4];poly_ecg=signal.resample_poly(stress,1,4,window=('kaiser',5.),padtype='constant')
ref_ecg=signal.resample_poly(clean,1,4,window=('kaiser',5.),padtype='constant')
ecg_rows={name:float(np.sqrt(np.mean((y[125:-125]-ref_ecg[125:-125])**2))) for name,y in [('naive',naive_ecg),('poly',poly_ecg)]}
check('E5.11: redukcja stres-testu EKG z antyaliasingiem zmniejsza RMSE',ecg_rows['poly']<ecg_rows['naive'])
RESULTS['E5.11']={'tone_amplitudes':amps,'FIR_gain_dB_at100_180':db_response(h100).tolist(),'ECG_RMSE_mV':ecg_rows,'ECG_alias_Hz':70}
save_figure(fig,FIG,'fig_05_11_decimation')


OK: E5.11: alias 180 Hz = −20 Hz i amplituda 0,4
OK: E5.11: jawny antyaliasing zachowuje ton20 Hz
OK: E5.11: resample_poly zachowuje ton20 Hz
OK: E5.11: redukcja stres-testu EKG z antyaliasingiem zmniejsza RMSE
Figura: figures/generated/fig_05_11_decimation.png (+ SVG)


## E5.12 — obrazy widma i nieodwracalna utrata informacji; rys. 5.12

Najpierw 15 Hz, A=1, fs100 Hz, 3 s; wstawienie trzech zer i interpolacja do400 Hz.
Widmo wstawionych zer zawiera obrazy przy 85,115,185 Hz; filtr interpolacyjny
ma wzmocnienie up=4 w paśmie, aby zachować amplitudę. Funkcja resample_poly
realizuje ten krok wewnętrznie. Drugi model: 70 Hz przy100 Hz jest identyczne
z -sin(30 Hz). Interpolacja x4 nie rozróżnia obu źródeł. Ocena po pominięciu0.1 s.
Metadane: sprawdzenie 250→360 Hz (up36/down25), długości i siatki zdarzeń.

![Rys. 5.12](../figures/generated/fig_05_12_interpolation.png)


In [13]:
ts=np.arange(300)/100;xs=np.sin(2*np.pi*15*ts)
zero=np.zeros(4*len(xs));zero[::4]=xs
interp=signal.resample_poly(xs,4,1,window=('kaiser',5.),padtype='constant')
tu=np.arange(len(interp))/400
f=np.fft.rfftfreq(len(zero),1/400);a=2*abs(np.fft.rfft(zero))/len(zero)
ai=2*abs(np.fft.rfft(interp))/len(interp)
check('E5.12: wstawienie zer tworzy obraz85 Hz',a[np.argmin(abs(f-85))]>.24)
check('E5.12: interpolacja zachowuje amplitudę15 Hz',abs(tone_fit(interp[100:-100],400,15,.25)-1)<.01)
ts2=np.arange(100)/100;source=np.sin(2*np.pi*70*ts2)
check('E5.12: próbki70 i aliasu−30 Hz identyczne',np.allclose(source,-np.sin(2*np.pi*30*ts2),atol=1e-12))
up=signal.resample_poly(source,4,1,window=('kaiser',5.),padtype='constant');td=np.arange(400)/400
x70=np.sin(2*np.pi*70*td);x30=-np.sin(2*np.pi*30*td)
err70=float(np.sqrt(np.mean((up[40:-40]-x70[40:-40])**2)))
err30=float(np.sqrt(np.mean((up[40:-40]-x30[40:-40])**2)))
check('E5.12: interpolacja odtwarza alias, nie utracone70 Hz',err30<.01 and err70>.9)
fig,ax=plt.subplots(3,1,figsize=(9,7.5),layout='constrained')
mask=(tu>=1)&(tu<=1.12)
ax[0].plot(tu[mask],zero[mask],'o:',ms=3,label='Po wstawieniu zer')
ax[0].plot(tu[mask],interp[mask],'-',label='Po filtrze interpolacyjnym')
ax[0].set(xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='a) Zera są etapem pośrednim')
ax[0].legend()
ax[1].plot(f,a,label='Wstawione zera');ax[1].plot(f,ai,'--',label='Interpolacja x4')
ax[1].set(xlim=(0,200),xlabel='Częstotliwość [Hz]',ylabel='Amplituda [j.u.]',title='b) Usunięcie obrazów i przywrócenie skali amplitudy')
ax[1].legend()
m=(td>=.1)&(td<=.24)
ax[2].plot(td[m],x70[m],':',label='Pierwotne 70 Hz');ax[2].plot(td[m],x30[m],'-',label='Alias −30 Hz');ax[2].plot(td[m],up[m],'--',label='Interpolacja danych 100 Hz')
ax[2].set(xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='c) Więcej próbek nie odtwarza wcześniejszego źródła')
ax[2].legend(fontsize=7)
out=signal.resample_poly(np.zeros(1001),36,25,window=('kaiser',5.),padtype='constant')
check('E5.12: długość wyniku wymiernego = ceil(N*up/down)',len(out)==(1001*36+24)//25)
new_fs=250*36/25;new_t=np.arange(len(out))/new_fs
check('E5.12: nowa siatka co1/fs_out',np.allclose(np.diff(new_t),1/360))
new_indices=np.rint(events*250).astype(int)
check('E5.12: błąd zaokrąglenia adnotacji <= połowa próbki',np.max(abs(new_indices/250-events))<=1/500+1e-12)
RESULTS['E5.12']={'RMSE_interpolation_vs70':err70,'RMSE_interpolation_vs_alias30':err30,
    'rational_input_samples':1001,'rational_output_samples':len(out),'rational_fs_out':new_fs,
    'events_at250Hz_first5':new_indices[:5].tolist(),'times_s_first5':events[:5].tolist()}
save_figure(fig,FIG,'fig_05_12_interpolation')


OK: E5.12: wstawienie zer tworzy obraz85 Hz
OK: E5.12: interpolacja zachowuje amplitudę15 Hz
OK: E5.12: próbki70 i aliasu−30 Hz identyczne
OK: E5.12: interpolacja odtwarza alias, nie utracone70 Hz
OK: E5.12: długość wyniku wymiernego = ceil(N*up/down)
OK: E5.12: nowa siatka co1/fs_out
OK: E5.12: błąd zaokrąglenia adnotacji <= połowa próbki
Figura: figures/generated/fig_05_12_interpolation.png (+ SVG)


## E5.13 — audyt kompletnego preprocessingu; rys. 5.13

Te same 12 s EKG i seed2026, bez dodatkowego stres-testu180 Hz.
P0: sam resampling. P1: FIR1001 Hamming BP0.7–35, filtfilt odd/padlen3003 +
notch50 Q30 sosfiltfilt odd/padlen9. P2: Butterworth BP N4, sosfiltfilt
odd/padlen27 + ten notch. P3: Czebyszew I BP N4, rp0.5 dB, taki sam tryb/notch.
P4: P2 bez notcha. P5: przyczynowe sekcje Butterwortha i notcha ze stanem zerowym,
potem **offline** resample_poly — nie jest to kompletny system online.
Wszystkie kończą resample_poly 1/4 (Kaiser5, constant); fs_out250 Hz.
Odniesienie: czysty sygnał poddany wyłącznie temu samemu resamplingowi.

Cechy R/QRS: pominięto po1 s, lokalne wyszukiwanie maksimum ±80 ms,
okno QRS ±80 ms po wyrównaniu do własnego maksimum. Miary amplitudy względem zera,
bez odejmowania lokalnej linii bazowej i bez dopasowywania skali. PSD **błędu**:
Welch4 s, Hann okresowy,50% overlap, detrend constant; moc sumą PSD*df.
Pasmo niskie0.25–0.5 Hz,49–51 Hz,60–100 Hz; nie udajemy rozdzielczości0.05 Hz.

![Rys. 5.13](../figures/generated/fig_05_13_preprocessing_audit.png)


In [14]:
bn,an=signal.iirnotch(50,30,fs=FS);notch=signal.tf2sos(bn,an)
b_fir=signal.firwin(1001,[.7,35],pass_zero=False,fs=FS,window='hamming')
sb=signal.butter(4,[.7,35],btype='bandpass',fs=FS,output='sos')
sc=signal.cheby1(4,.5,[.7,35],btype='bandpass',fs=FS,output='sos')
def notch_fb(y):return signal.sosfiltfilt(notch,y,padtype='odd',padlen=9)
def resample(y):return signal.resample_poly(y,1,4,window=('kaiser',5.),padtype='constant')
outputs={
 'P0':resample(noisy),
 'P1':resample(notch_fb(signal.filtfilt(b_fir,[1.],noisy,padtype='odd',padlen=3003))),
 'P2':resample(notch_fb(signal.sosfiltfilt(sb,noisy,padtype='odd',padlen=27))),
 'P3':resample(notch_fb(signal.sosfiltfilt(sc,noisy,padtype='odd',padlen=27))),
 'P4':resample(signal.sosfiltfilt(sb,noisy,padtype='odd',padlen=27)),
 'P5':resample(signal.sosfilt(notch,signal.sosfilt(sb,noisy)))
}
reference=resample(clean);tout=np.arange(len(reference))/250
rows=[]
for name,y in outputs.items():
    check(f'E5.13: zgodna długość i skończony wynik {name}',len(y)==3000 and np.isfinite(y).all())
    metrics=preservation_metrics(reference,y,events,250)
    for key,lo,hi in [('low_0_25_0_5',.25,.5),('mains49_51',49,51),('high60_100',60,100)]:
        metrics[key+'_mV2']=band_power(y-reference,250,lo,hi)
    rows.append({'ID':name,**metrics})
byid={r['ID']:r for r in rows}
check('E5.13: notch zmniejsza błąd49–51 Hz względem pary bez notcha',byid['P2']['mains49_51_mV2']<byid['P4']['mains49_51_mV2'])
check('E5.13: przyczynowe filtry nadal wprowadzają opóźnienie',byid['P5']['R_time_signed_mean_ms']>=8)
check('E5.13: FIR zachowuje lepiej QRS niż wąskie pasmo A',byid['P1']['R_amplitude_error_pct']<RESULTS['E5.1']['A']['R_amplitude_error_pct'])
fig,ax=plt.subplots(3,1,figsize=(9.3,8.4),layout='constrained')
at=events[5]
ax[0].plot((tout-at)*1000,reference,color='black',lw=1.8,label='Odniesienie')
for name,ls in [('P1','-'),('P2','--'),('P3','-.'),('P5',':')]:ax[0].plot((tout-at)*1000,outputs[name],ls,label=name)
ax[0].set(xlim=(-100,150),xlabel='Czas względem zdarzenia [ms]',ylabel='Amplituda [mV]',title='a) Morfologia i czas po całej sekwencji')
ax[0].legend(ncol=5)
for name,ls in [('P0',':'),('P1','-'),('P2','--'),('P4','-.')]:
    f,p=psd(outputs[name]-reference,250)
    ax[1].semilogy(f,p,ls,label=name)
ax[1].set(xlim=(.25,105),ylim=(1e-12,.1),xlabel='Częstotliwość [Hz]',ylabel='PSD błędu [mV²/Hz]',title='b) Redukcja zakłócenia i koszt dla czystego sygnału sumują się w błędzie')
ax[1].legend(ncol=4)
xx=np.arange(len(rows));width=.24
for offset,key,label in [(-width,'RMSE_start_mV','Pierwsza 1 s'),(0,'RMSE_center_mV','Środek'),(width,'RMSE_end_mV','Ostatnia 1 s')]:
    ax[2].bar(xx+offset,[r[key] for r in rows],width,label=label)
ax[2].set_xticks(xx,[r['ID'] for r in rows]);ax[2].set(ylabel='RMSE [mV]',title='c) Oddzielna ocena obu brzegów')
ax[2].legend(ncol=3)
RESULTS['E5.13']=rows
print(json.dumps(rows,ensure_ascii=False,indent=2))
# Fragment tabeli do źródłowego rozdziału, bez ręcznego przepisywania liczb.
table='| Wariant | Błąd R [%] | MAE czasu R [ms] | NRMSE QRS | RMSE środek [mV] |\n|---|---:|---:|---:|---:|\n'
for r in rows:
    vals=[f"{r['R_amplitude_error_pct']:.2f}",f"{r['R_time_MAE_ms']:.2f}",f"{r['QRS_NRMSE_aligned']:.4f}",f"{r['RMSE_center_mV']:.4f}"]
    table+='| '+r['ID']+' | '+' | '.join(v.replace('.',',') for v in vals)+' |\n'
(ROOT/'data/ch05_pipeline_table.md').write_text(table, encoding='utf-8', newline='\n')
save_figure(fig,FIG,'fig_05_13_preprocessing_audit')


OK: E5.13: zgodna długość i skończony wynik P0
OK: E5.13: zgodna długość i skończony wynik P1
OK: E5.13: zgodna długość i skończony wynik P2
OK: E5.13: zgodna długość i skończony wynik P3
OK: E5.13: zgodna długość i skończony wynik P4
OK: E5.13: zgodna długość i skończony wynik P5
OK: E5.13: notch zmniejsza błąd49–51 Hz względem pary bez notcha
OK: E5.13: przyczynowe filtry nadal wprowadzają opóźnienie
OK: E5.13: FIR zachowuje lepiej QRS niż wąskie pasmo A
[
  {
    "ID": "P0",
    "R_amplitude_error_pct": 9.333161921309236,
    "R_time_MAE_ms": 2.0,
    "R_time_signed_mean_ms": -0.6666666666666666,
    "R_time_signed_SD_ms": 2.748737083745107,
    "QRS_NRMSE_aligned": 0.12319262555803728,
    "beats": 12,
    "RMSE_center_mV": 0.14460360513749354,
    "RMSE_start_mV": 0.20075045753464696,
    "RMSE_end_mV": 0.14951603430446092,
    "low_0_25_0_5_mV2": 0.013436988739963377,
    "mains49_51_mV2": 0.003223832884172229,
    "high60_100_mV2": 6.263030569092056e-05
  },
  {
    "ID": "P1",


## Wyniki i zakres wykonanej kontroli

Ostatnia komórka zapisuje liczby do `data/ch05_results.json` i wersje środowiska
do `chapter_notes/ch05_environment.json`. Kontrole dotyczą znanych modeli,
skalowania, czasu, stanu i zgodności operacji. Nie dowodzą skuteczności klinicznej
ani jakości pełnego detektora. Wszystkie obrazy pozostają osobnymi plikami.


In [15]:
for i in range(1,14):
    for ext in ['png','svg']:
        check(f'Figura5.{i} {ext}',len(list(FIG.glob(f'fig_05_{i:02d}_*.{ext}')))==1)
RESULTS['audit']={'passed_checks':len(CHECKS),'checks':CHECKS,'experiments':13,'figures':13,'external_data':False}
(ROOT/'data/ch05_results.json').write_text(json.dumps(RESULTS,ensure_ascii=False,indent=2,allow_nan=False)+'\n', encoding='utf-8', newline='\n')
(ROOT/'chapter_notes/ch05_environment.json').write_text(json.dumps(ENV,ensure_ascii=False,indent=2)+'\n', encoding='utf-8', newline='\n')
print('Zakończono; liczba kontroli:',len(CHECKS))


OK: Figura5.1 png
OK: Figura5.1 svg
OK: Figura5.2 png
OK: Figura5.2 svg
OK: Figura5.3 png
OK: Figura5.3 svg
OK: Figura5.4 png
OK: Figura5.4 svg
OK: Figura5.5 png
OK: Figura5.5 svg
OK: Figura5.6 png
OK: Figura5.6 svg
OK: Figura5.7 png
OK: Figura5.7 svg
OK: Figura5.8 png
OK: Figura5.8 svg
OK: Figura5.9 png
OK: Figura5.9 svg
OK: Figura5.10 png
OK: Figura5.10 svg
OK: Figura5.11 png
OK: Figura5.11 svg
OK: Figura5.12 png
OK: Figura5.12 svg
OK: Figura5.13 png
OK: Figura5.13 svg
Zakończono; liczba kontroli: 86
